In [1]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)

from nltk.sentiment.vader import SentimentIntensityAnalyzer
import unicodedata

In [2]:
# Reading the saved data pickle file
df_articles = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/PressReleaseDate.pkl', compression=None)

In [3]:
df_articles

,Date,Articles,Title,Text,PressReleaseCount,Company
2019-09-25,2019-09-25,. INCYTE ANNOUNCES FIRST PATIENT TREATED IN PH...,. INCYTE ANNOUNCES FIRST PATIENT TREATED IN PH...,. INCYTE ANNOUNCES FIRST PATIENT TREATED IN PH...,13,HLT
2019-09-26,2019-09-26,. SEMPRA ENERGY'S SUBSIDIARY IENOVA SIGNS TWO ...,. SEMPRA ENERGY'S SUBSIDIARY IENOVA SIGNS TWO ...,. SEMPRA ENERGY'S SUBSIDIARY IENOVA SIGNS TWO ...,11,MRNA
2019-09-27,2019-09-27,. BNY MELLON SAYS THOMAS GIBBONS APPOINTED INT...,. BNY MELLON SAYS THOMAS GIBBONS APPOINTED INT...,. BNY MELLON SAYS THOMAS GIBBONS APPOINTED INT...,7,ICE
2019-09-30,2019-09-30,. SEMPRA ENERGY TO SELL STAKE IN LUZ DEL SUR I...,. SEMPRA ENERGY TO SELL STAKE IN LUZ DEL SUR I...,. SEMPRA ENERGY TO SELL STAKE IN LUZ DEL SUR I...,10,ABT
2019-10-01,2019-10-01,. CIGNA EXPANDS MEDICARE ADVANTAGE OFFERINGS A...,. CIGNA EXPANDS MEDICARE ADVANTAGE OFFERINGS A...,. CIGNA EXPANDS MEDICARE ADVANTAGE OFFERINGS A...,12,LKQ
...,...,...,...,...,...,...
2021-08-19,2021-08-19,. TYLER TECHNOLOGIES EXTENDS 22-YEAR CONTRACT ...,. TYLER TECHNOLOGIES EXTENDS 22-YEAR CONTRACT ...,. TYLER TECHNOLOGIES EXTENDS 22-YEAR CONTRACT ...,25,HD
2021-08-20,2021-08-20,. ELECTRONIC ARTS RELEASES EA SPORTS MADDEN NF...,. ELECTRONIC ARTS RELEASES EA SPORTS MADDEN NF...,. ELECTRONIC ARTS RELEASES EA SPORTS MADDEN NF...,13,HBAN
2021-08-23,2021-08-23,. T-MOBILE IS GIVING CUSTOMERS ONE YEAR OF APP...,. T-MOBILE IS GIVING CUSTOMERS ONE YEAR OF APP...,. T-MOBILE IS GIVING CUSTOMERS ONE YEAR OF APP...,29,HII
2021-08-24,2021-08-24,. SYNOPSYS ENABLES FIRST-PASS SILICON SUCCESS ...,. SYNOPSYS ENABLES FIRST-PASS SILICON SUCCESS ...,. SYNOPSYS ENABLES FIRST-PASS SILICON SUCCESS ...,30,INTU


In [4]:
df  = pd.DataFrame()

# Adding new columns to the data frame
df[['compound']] = ''
df[['neg']] = ''
df[['neu']] = ''
df[['pos']] = ''
df[['sentiment']] = ''

In [5]:
debug=False
sid = SentimentIntensityAnalyzer()
countNeg=0
countNeu=0
countPos=0
countNegWord=0
countNeuWord=0
countPosWord=0
def clean_text(text):
    '''Make text lowercase, remove text in square brackets,remove links,remove punctuation
    and remove words containing numbers.'''
    text = str(text).lower()
    text = re.sub('\[.*?\]', '', text)
    text = re.sub('https?://\S+|www\.\S+', '', text)
    text = re.sub('<.*?>+', '', text)
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub('\n', '', text)
    text = re.sub('\w*\d\w*', '', text)
    text.replace("  "," ")
    return text

for date, row in df_articles.T.iteritems():
    # print("The date is:"+str(date))
    # try:
    print(type(df_articles.loc[date, 'Articles']))
    sentence = unicodedata.normalize('NFKD', df_articles.loc[date, 'Articles'])#.encode('ascii','ignore').decode('ascii')
    sentence = clean_text(sentence)

    ss = sid.polarity_scores(sentence)

    #print(sentence)
    #print("#2")
    df.at[date, 'compound'] = ss['compound']
    df.at[date, 'neg'] = ss['neg']
    df.at[date, 'neu'] = ss['neu']
    df.at[date, 'pos'] = ss['pos']
    if ((ss['pos'] + ss['neg']) != 0):
        number = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
        if (number != ''):
            df.at[date, 'sentiment'] = 100*(ss['pos'] / (ss['pos'] + ss['neg']))
            # df.at[date, 'sentiment_complementary'] = 100 - 100*(ss['pos'] / (ss['pos'] + ss['neg']))

    countNeg+=ss['neg']
    countNeu+=ss['neu']
    countPos+=ss['pos']
        
    # except TypeError:
    #     #print("sentence is")
    #     #print(sentence)
    #     #print(df_vix.loc[date, 'articles'])
    #     print("ERROR:"+sentence)
    #     df.at[date, 'compound'] = 0
    #     df.at[date, 'neg'] = 0
    #     df.at[date, 'neu'] = 0
    #     df.at[date, 'pos'] = 0
    #     # df.at[date, 'sentiment'] = 0
    #     # df.at[date, 'sentiment_complementary'] = 0

<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class 'str'>
<class

In [6]:
df

,compound,neg,neu,pos,sentiment
2019-09-25,0.9972,0.006,0.898,0.096,94.117647
2019-09-26,0.991,0.018,0.872,0.109,85.826772
2019-09-27,0.9879,0.006,0.89,0.104,94.545455
2019-09-30,0.9964,0.012,0.879,0.108,90.0
2019-10-01,0.9973,0.0,0.86,0.14,100.0
...,...,...,...,...,...
2021-08-19,0.9987,0.046,0.838,0.117,71.779141
2021-08-20,0.9967,0.042,0.835,0.123,74.545455
2021-08-23,0.9994,0.042,0.834,0.124,74.698795
2021-08-24,0.9998,0.027,0.816,0.157,85.326087


In [7]:
print("Number of Negative words="+str(countNeg))
print("Number of Positive words="+str(countPos))
print("Number of Neutral words="+str(countNeu))

df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticles.pkl")

Number of Negative words=12.380999999999986
Number of Positive words=51.961999999999996
Number of Neutral words=395.64400000000023


In [8]:
df = df.join(df_articles)

In [9]:
df

,compound,neg,neu,pos,sentiment,Date,Articles,Title,Text,PressReleaseCount,Company
2019-09-25,0.9972,0.006,0.898,0.096,94.117647,2019-09-25,. INCYTE ANNOUNCES FIRST PATIENT TREATED IN PH...,. INCYTE ANNOUNCES FIRST PATIENT TREATED IN PH...,. INCYTE ANNOUNCES FIRST PATIENT TREATED IN PH...,13,HLT
2019-09-26,0.991,0.018,0.872,0.109,85.826772,2019-09-26,. SEMPRA ENERGY'S SUBSIDIARY IENOVA SIGNS TWO ...,. SEMPRA ENERGY'S SUBSIDIARY IENOVA SIGNS TWO ...,. SEMPRA ENERGY'S SUBSIDIARY IENOVA SIGNS TWO ...,11,MRNA
2019-09-27,0.9879,0.006,0.89,0.104,94.545455,2019-09-27,. BNY MELLON SAYS THOMAS GIBBONS APPOINTED INT...,. BNY MELLON SAYS THOMAS GIBBONS APPOINTED INT...,. BNY MELLON SAYS THOMAS GIBBONS APPOINTED INT...,7,ICE
2019-09-30,0.9964,0.012,0.879,0.108,90.0,2019-09-30,. SEMPRA ENERGY TO SELL STAKE IN LUZ DEL SUR I...,. SEMPRA ENERGY TO SELL STAKE IN LUZ DEL SUR I...,. SEMPRA ENERGY TO SELL STAKE IN LUZ DEL SUR I...,10,ABT
2019-10-01,0.9973,0.0,0.86,0.14,100.0,2019-10-01,. CIGNA EXPANDS MEDICARE ADVANTAGE OFFERINGS A...,. CIGNA EXPANDS MEDICARE ADVANTAGE OFFERINGS A...,. CIGNA EXPANDS MEDICARE ADVANTAGE OFFERINGS A...,12,LKQ
...,...,...,...,...,...,...,...,...,...,...,...
2021-08-19,0.9987,0.046,0.838,0.117,71.779141,2021-08-19,. TYLER TECHNOLOGIES EXTENDS 22-YEAR CONTRACT ...,. TYLER TECHNOLOGIES EXTENDS 22-YEAR CONTRACT ...,. TYLER TECHNOLOGIES EXTENDS 22-YEAR CONTRACT ...,25,HD
2021-08-20,0.9967,0.042,0.835,0.123,74.545455,2021-08-20,. ELECTRONIC ARTS RELEASES EA SPORTS MADDEN NF...,. ELECTRONIC ARTS RELEASES EA SPORTS MADDEN NF...,. ELECTRONIC ARTS RELEASES EA SPORTS MADDEN NF...,13,HBAN
2021-08-23,0.9994,0.042,0.834,0.124,74.698795,2021-08-23,. T-MOBILE IS GIVING CUSTOMERS ONE YEAR OF APP...,. T-MOBILE IS GIVING CUSTOMERS ONE YEAR OF APP...,. T-MOBILE IS GIVING CUSTOMERS ONE YEAR OF APP...,29,HII
2021-08-24,0.9998,0.027,0.816,0.157,85.326087,2021-08-24,. SYNOPSYS ENABLES FIRST-PASS SILICON SUCCESS ...,. SYNOPSYS ENABLES FIRST-PASS SILICON SUCCESS ...,. SYNOPSYS ENABLES FIRST-PASS SILICON SUCCESS ...,30,INTU


In [10]:
df.to_pickle("/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentimentArticles.pkl")